In [2]:
import pandas as pd
import altair as alt
import numpy as np

DATA_PATH = 'final_clean_merged_owid.csv' 
x = pd.read_csv(DATA_PATH, parse_dates=['year'])
alt.data_transformers.enable('default', max_rows=20000)

DataTransformerRegistry.enable('default')

___
## **Section 1 - Column Selection & Renaming**

To streamline analysis, the dataset is filtered to include only the variables relevant to the two analytic questions:
- Reserve-related attributes for computing *gold_share* (Q1).  
- Social and economic indicators for assessing relationships with total reserves (Q2).  

The resulting dataframe retains only 12 columns, balancing completeness with interpretability. The dataset has also been filtered to only include dates where majority of the gold reserve data is available (2005-2015)


In [3]:
relevant_cols = [
    'country_x',
    'iso_code',
    'year',
    'region',
    'income_group',
    'Total reserves (includes gold, current US$)',
    'Total reserves minus gold (current US$)',
    'income',
    'life_expectancy',
    'years_in_school_men',
    'years_in_school_women',
    'Access to electricity (% of population)'
]

df = x[relevant_cols]
df = df[df['year'].dt.year.between(2005,2015)]
df = df.rename(columns = {'country_x':'country',
                        'Total reserves (includes gold, current US$)':'reserves_gold', 
                        'Total reserves minus gold (current US$)':'reserves_no_gold',
                        'Access to electricity (% of population)':'elec_access',
                        })
print('Filtered dataset shape:', df.shape)
print('Columns retained:\n', list(df.columns))

Filtered dataset shape: (2413, 12)
Columns retained:
 ['country', 'iso_code', 'year', 'region', 'income_group', 'reserves_gold', 'reserves_no_gold', 'income', 'life_expectancy', 'years_in_school_men', 'years_in_school_women', 'elec_access']


___
## **Section 2 – Derived Columns**

This section derives two new variables essential for the analytic questions:

- `gold_share` - the proportion of gold in a nation’s total reserves.  
  $$\text{gold\_share} = 
  \frac{(\text{Total reserves (includes gold)} - \text{Total reserves (without gold)})}
       {\text{Total reserves (includes gold)}} \times 100$$

- `education_index` - the average years of schooling between men and women.  
  $$\text{education\_index} = 
  \frac{\text{Years in school (men)} + \text{Years in school (women)}}{2}$$

- `social_index` - blend of 3 social factors: Education Index, Access to Electricity and Life Expectancy
  $$\text{Social Index} = 
  \frac{\text{Life Expectancy}_{\text{norm}} + \text{Electricity Access}_{\text{norm}} + \text{Education Index}_{\text{norm}}}{3}$$
  $$\text{[Variable]}_{\text{norm}} = 
  \frac{\text{[Variable]} - \min(\text{[Variable]})}{\max(\text{[Variable]}) - \min(\text{[Variable]})}

Both derived columns enable meaningful cross-country and regional comparisons for subsequent analysis.




In [4]:
# Calculate the percentage of total reserves held as gold
df['gold_share'] = ( ( df['reserves_gold'] - df['reserves_no_gold'] ) / df['reserves_gold'] ) * 100
# Handle non-sensical percentage values by setting them to missing
df.loc[ ( df['gold_share'] < 0 ) | ( df['gold_share'] > 100 ) , 'gold_share'] = np.nan

# Calculate the simple average of years in school for both genders
df['education_index'] = ( df['years_in_school_men'] + df['years_in_school_women'] ) / 2

# Remove rows with calculated missing values for gold share or education index
df = df[ ( ~df['gold_share'].isna() ) & ( ~df['education_index'].isna() ) ]

# List the columns required for calculating the composite Social Index
columns_to_normalize = [ 'life_expectancy' , 'elec_access' , 'education_index' ]

# Determine the min and max values for normalization
min_values = df[columns_to_normalize].min()
max_values = df[columns_to_normalize].max()

normalized_columns = []

# Normalize each component column to a 0-1 range (Min-Max scaling)
for col in columns_to_normalize:
    new_col_name = f'{col}_norm'
    df[new_col_name] = ( df[col] - min_values[col] ) / ( max_values[col] - min_values[col] )
    normalized_columns.append(new_col_name)

# Calculate the Social Index as the mean of the three normalized components
df['social_index'] = df[normalized_columns].mean(axis = 1)

# Apply logarithmic transformations for key variables to handle skewness
df['gold_share_log'] = np.log(df['gold_share'])
df['log_reserves_gold'] = np.log1p(df['reserves_gold'])
df['log_reserves_no_gold'] = np.log1p(df['reserves_no_gold'])

c:\Users\manan\miniconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


___
# **View 2**

Interactions Used:
- Slider
    - Year
- Bi-Directional Selection Interval
    - Bi-Directional Interaction between 


In [5]:
# - Define interactive selection parameters for View 2
year_slider2 = alt.binding_range(min = 2005, max = 2015, step = 1, name = 'Year' ) # Defines year range for View 2 slider
slider_selection2 = alt.selection_point(bind = year_slider2, fields = [ 'year' ], value = { 'year' : 2015 }) # Selection for View 2 slider
brush_selection = alt.selection_interval(encodings = [ 'x' , 'y' ], value = { 'x' : [ 0.9 , 1 ], 'y' : [ 0 , 30 ]}) # Defines interval brush selection

# - Chart 1 Reserves vs. Social Index
chart1_v2 = alt.Chart(df).mark_circle(opacity = 0.6).add_params(slider_selection2, brush_selection).transform_calculate(
    # Calculate year from date field
    year = "year(datum['year'])"
).transform_filter(slider_selection2).encode(
    alt.X('social_index:Q', title = 'Social Index (0-1)'),
    alt.Y('log_reserves_gold:Q', title = 'Total Reserves'),
    alt.Size('income:Q', scale = alt.Scale(range = [ 10 , 100 ]), title = 'GDP Per Capita'),
    alt.Color('region:N', title = 'Region'),
    opacity = alt.condition(brush_selection, alt.value(0.75), alt.value(0.3)), # Dim points outside brush area
    tooltip = [ 'country' , 'social_index' , 'log_reserves_gold']
).properties(
    width = 950,
    height = 400,
    title = 'Total Reserves V Social Index'
)

# - Chart 2: Gold Share vs. GDP 
chart2_v2 = alt.Chart(df).mark_circle().encode(
    alt.Y('gold_share:Q', title = 'Gold Share (%)'),
    alt.X('income:Q', title = 'GDP Per Capita'),
    color = alt.when(brush_selection).then('region:N', legend = None).otherwise(alt.value('lightgrey')), # Color points inside brush area
    tooltip = [ 'country:N' , 'gold_share:Q' , 'income:Q' ]
).add_params(brush_selection).properties(
    width = 450,
    height = 400,
    title = 'Gold Share V GDP Per Capita'
)

# - Chart 3 Education Years by Gender (Stacked Bar Plot)
chart3_v2 = alt.Chart(df).transform_calculate(
    # Calculate year from date field
    year = "year(datum['year'])",
    # Calculate education difference for tooltip
    school_difference = "datum.years_in_school_men - datum.years_in_school_women"
).add_params(slider_selection2, brush_selection).transform_filter(slider_selection2, brush_selection).transform_fold(
    # Restructure data for stacked bars
    [ 'years_in_school_men' , 'years_in_school_women' ],
    as_ = [ 'Gender' , 'Years_in_School' ]
).transform_impute(
    # Impute missing values to 0 for plotting
    impute = 'Years_in_School',
    key = 'country',
    value = 0
).mark_bar().encode(
    alt.X('Years_in_School:Q', title = 'Total Average Years in School'),
    alt.Y('country:N', title = 'Country', sort = alt.EncodingSortField(field = 'Years_in_School', op = 'sum', order = 'descending')),
    alt.Color('Gender:N', title = 'Gender'),
    tooltip = [
        'country:N',
        'Gender:N',
        { 'field' : 'Years_in_School', 'type' : 'quantitative', 'title' : 'Years in School' },
        alt.Tooltip('school_difference:Q', title = 'Education Difference (M-W)')
    ],
).properties(
    title = 'Average Years in School by Gender and Country (Stacked)',
    width = 450,
    height = alt.Step(10),
)

# - Combine the charts into a final dashboard View 2 
view2 = ( chart1_v2 & ( chart2_v2 | chart3_v2 ) ).resolve_scale(
    color = 'independent', # Allow colors to be different across combined charts
    size = 'independent' # Allow sizes to be different across combined charts
).properties(title = alt.TitleParams(
    text = 'View 2',
    anchor = 'middle',
    fontSize = 30 
    )
)

view2

alt.VConcatChart(...)